# Intrinsic Lineage Tree Visualization

This notebook walks through the lineage / phylogeny visualization workflow
for `IntrinsicEvolutionExperiment` runs using the
`intrinsic_gene_snapshots.jsonl` artifact.

## What this notebook covers

1. **Load** `intrinsic_gene_snapshots.jsonl` produced by a run
2. **Build** the lineage DAG (two-parent reproduction makes it a DAG, not a
   strict tree)
3. **Plot** the lineage tree with optional per-gene chromosome colouring
4. **Compute** summary statistics:
   - Surviving-lineage count over time
   - Lineage depth over time
5. **Inspect** raw chromosome trajectories for individual lineages

### Data source

Set `SNAPSHOT_PATH` to the path of your
`intrinsic_gene_snapshots.jsonl` file **or** the directory that contains it.
The cell below creates a small synthetic fixture so the notebook runs
out-of-the-box without a real experiment on disk.

In [ ]:
import json
import pathlib
import tempfile

# ── Configuration ──────────────────────────────────────────────────────────
# Change SNAPSHOT_PATH to point at a real run directory, e.g.:
#   SNAPSHOT_PATH = "experiments/intrinsic_evolution_smoke"
# or directly to the file:
#   SNAPSHOT_PATH = "experiments/intrinsic_evolution_smoke/intrinsic_gene_snapshots.jsonl"
SNAPSHOT_PATH = None  # None → use the synthetic fixture created below

# ── Synthetic fixture (skip when SNAPSHOT_PATH is set) ─────────────────────
if SNAPSHOT_PATH is None:
    _tmp = tempfile.mkdtemp(prefix="lineage_nb_")
    _fixture = pathlib.Path(_tmp) / "intrinsic_gene_snapshots.jsonl"

    def _agent(agent_id, parent_ids, generation=0, lr=0.01, gamma=0.99):
        return {
            "agent_id": agent_id,
            "agent_type": "system",
            "generation": generation,
            "parent_ids": parent_ids,
            "chromosome": {"learning_rate": lr, "gamma": gamma},
        }

    snapshots = [
        # Step 0 – two founder lineages, 4 agents
        {"step": 0, "agents": [
            _agent("r1", [], lr=0.005, gamma=0.95),
            _agent("r2", [], lr=0.020, gamma=0.99),
            _agent("r3", [], lr=0.010, gamma=0.97),
            _agent("r4", [], lr=0.015, gamma=0.98),
        ]},
        # Step 50 – r2 lineage expands
        {"step": 50, "agents": [
            _agent("r1", [], generation=0, lr=0.005),
            _agent("r2", [], generation=0, lr=0.020),
            _agent("c1", ["r2"], generation=1, lr=0.018),
            _agent("c2", ["r2"], generation=1, lr=0.022),
            _agent("r3", [], generation=0, lr=0.010),
            # r4 gone
        ]},
        # Step 100 – r1 lineage extinct; r2 lineage deeper; dual-parent child
        {"step": 100, "agents": [
            _agent("r2",  [], generation=0, lr=0.020),
            _agent("c1",  ["r2"], generation=1, lr=0.018),
            _agent("c2",  ["r2"], generation=1, lr=0.022),
            _agent("gc1", ["c1", "c2"], generation=2, lr=0.020),  # crossover child
            _agent("r3",  [], generation=0, lr=0.010),
            _agent("c3",  ["r3"], generation=1, lr=0.011),
        ]},
        # Step 150 – final snapshot
        {"step": 150, "agents": [
            _agent("r2",  [], generation=0, lr=0.020),
            _agent("gc1", ["c1", "c2"], generation=2, lr=0.021),
            _agent("ggc1", ["gc1"], generation=3, lr=0.021),
            _agent("r3",  [], generation=0, lr=0.010),
            _agent("c3",  ["r3"], generation=1, lr=0.012),
        ]},
    ]

    with _fixture.open("w") as fh:
        for snap in snapshots:
            fh.write(json.dumps(snap) + "\n")

    SNAPSHOT_PATH = str(_fixture)
    print(f"Using synthetic fixture at {SNAPSHOT_PATH}")
else:
    print(f"Using snapshot file: {SNAPSHOT_PATH}")

## 1. Load the snapshots

In [ ]:
from farm.analysis.phylogenetics import (
    load_intrinsic_snapshots,
    flatten_snapshots_to_agent_records,
    build_intrinsic_lineage_dag,
    extract_chromosomes_from_snapshots,
    compute_surviving_lineage_count_over_time,
    compute_lineage_depth_over_time,
    plot_intrinsic_lineage_tree,
)

snapshots = load_intrinsic_snapshots(SNAPSHOT_PATH)
print(f"Loaded {len(snapshots)} snapshot steps")
steps = sorted(s['step'] for s in snapshots)
print(f"Steps: {steps}")
print(f"Agents at final step: {len(snapshots[-1]['agents'])}")

## 2. Build the lineage DAG

In [ ]:
tree = build_intrinsic_lineage_dag(SNAPSHOT_PATH)
summary = tree.summary()

print(f"Nodes  : {summary.num_nodes}")
print(f"Founders (roots): {summary.num_founders}  → {tree.roots}")
print(f"Max depth: {summary.max_depth}")
print(f"Mean depth: {summary.mean_depth:.2f}")
print(f"Mean branching factor: {summary.mean_branching_factor:.2f}")
print(f"Is DAG (dual-parent): {tree.is_dag}")
print(f"Orphan nodes: {summary.num_orphans}")

## 3. Plot – lineage colouring

In [ ]:
import pathlib, tempfile
from farm.analysis.common.context import AnalysisContext
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # non-interactive; change to "inline" in Jupyter

out_dir = pathlib.Path(tempfile.mkdtemp(prefix="lineage_plots_"))
ctx = AnalysisContext(output_path=out_dir)

out_path = plot_intrinsic_lineage_tree(
    tree, ctx,
    color_by_lineage=True,
    title="Intrinsic Lineage Tree – coloured by founder lineage",
)
print(f"Saved: {out_path}")
from IPython.display import Image
Image(str(out_path))

## 4. Plot – gene-value colouring

Nodes are coloured by the `learning_rate` chromosome value on a continuous
`viridis` scale.  Founders that have no chromosome entry (e.g. synthetic
"seed" parents) appear at the midpoint colour.

In [ ]:
chromosomes = extract_chromosomes_from_snapshots(snapshots)
print(f"Chromosome entries: {len(chromosomes)}")

out_path_gene = plot_intrinsic_lineage_tree(
    tree, ctx,
    gene="learning_rate",
    chromosomes=chromosomes,
    title="Intrinsic Lineage Tree – coloured by learning_rate",
)
print(f"Saved: {out_path_gene}")
Image(str(out_path_gene))

## 5. Surviving-lineage count over time

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

lineage_counts = compute_surviving_lineage_count_over_time(tree, snapshots)
print("Step  →  Surviving lineages")
for step, count in lineage_counts:
    print(f"  {step:>5}  →  {count}")

steps_lc, counts_lc = zip(*lineage_counts) if lineage_counts else ([], [])
fig, ax = plt.subplots(figsize=(7, 3))
ax.step(steps_lc, counts_lc, where="post", lw=2)
ax.set_xlabel("Simulation step")
ax.set_ylabel("Distinct founder lineages")
ax.set_title("Surviving lineage count over time")
ax.yaxis.get_major_locator().set_params(integer=True)
plt.tight_layout()
lc_path = out_dir / "lineage_count_over_time.png"
fig.savefig(lc_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {lc_path}")
Image(str(lc_path))

## 6. Lineage depth over time

In [ ]:
depth_series = compute_lineage_depth_over_time(tree, snapshots)
print("Step  →  max_depth / mean_depth")
for step, max_d, mean_d in depth_series:
    print(f"  {step:>5}  →  max={max_d}  mean={mean_d:.2f}")

if depth_series:
    steps_d, max_depths, mean_depths = zip(*depth_series)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(steps_d, max_depths, label="max depth", lw=2)
    ax.plot(steps_d, mean_depths, label="mean depth", lw=2, linestyle="--")
    ax.set_xlabel("Simulation step")
    ax.set_ylabel("Lineage depth")
    ax.set_title("Lineage depth of alive agents over time")
    ax.legend()
    plt.tight_layout()
    depth_path = out_dir / "lineage_depth_over_time.png"
    fig.savefig(depth_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {depth_path}")
    Image(str(depth_path))

## 7. Inspect individual lineages

Trace which chromosome values were observed along the ancestry path of a
specific agent.

In [ ]:
from farm.analysis.phylogenetics.intrinsic_loader import _trace_to_founder

FOCUS_AGENT = tree.roots[0] if tree.roots else None
if FOCUS_AGENT:
    print(f"Founder of all inspected agents: {FOCUS_AGENT}")

# Walk every alive agent at the final step back to its founder
final_agents = [a["agent_id"] for a in snapshots[-1].get("agents", [])]
print("\nAgent  →  founder  →  learning_rate")
for aid in final_agents:
    founder = _trace_to_founder(tree, str(aid))
    lr = chromosomes.get(str(aid), {}).get("learning_rate", "n/a")
    print(f"  {aid:<12}  →  {founder:<12}  →  {lr}")

## 8. Export tree as JSON / Newick

The underlying `PhylogeneticTree` object supports JSON and Newick export for
use with external tree viewers.

In [ ]:
import json, pathlib

json_path = out_dir / "lineage_tree.json"
json_path.write_text(tree.to_json())
print(f"JSON tree saved to: {json_path}")

newick_str = tree.to_newick()
newick_path = out_dir / "lineage_tree.newick"
newick_path.write_text(newick_str)
print(f"Newick string: {newick_str[:200]}{'...' if len(newick_str) > 200 else ''}")
print(f"Newick saved to: {newick_path}")